# Assignment 1

In this assignment, you'll be working with messy medical data and using regex to extract relevant infromation from the data. 

Each line of the `dates.txt` file corresponds to a medical note. Each note has a date that needs to be extracted, but each date is encoded in one of many formats.

The goal of this assignment is to correctly identify all of the different date variants encoded in this dataset and to properly normalize and sort the dates. 

Here is a list of some of the variants you might encounter in this dataset:
* 04/20/2009; 04/20/09; 4/20/09; 4/3/09
* Mar-20-2009; Mar 20, 2009; March 20, 2009;  Mar. 20, 2009; Mar 20 2009;
* 20 Mar 2009; 20 March 2009; 20 Mar. 2009; 20 March, 2009
* Mar 20th, 2009; Mar 21st, 2009; Mar 22nd, 2009
* Feb 2009; Sep 2009; Oct 2010
* 6/2008; 12/2009
* 2009; 2010

Once you have extracted these date patterns from the text, the next step is to sort them in ascending chronological order accoring to the following rules:
* Assume all dates in xx/xx/xx format are mm/dd/yy
* Assume all dates where year is encoded in only two digits are years from the 1900's (e.g. 1/5/89 is January 5th, 1989)
* If the day is missing (e.g. 9/2009), assume it is the first day of the month (e.g. September 1, 2009).
* If the month is missing (e.g. 2010), assume it is the first of January of that year (e.g. January 1, 2010).
* Watch out for potential typos as this is a raw, real-life derived dataset.

With these rules in mind, find the correct date in each note and return a pandas Series in chronological order of the original Series' indices. **This Series should be sorted by a tie-break sort in the format of ("extracted date", "original row number").**

For example if the original series was this:

    0    1999
    1    2010
    2    1978
    3    2015
    4    1985

Your function should return this:

    0    2
    1    4
    2    0
    3    1
    4    3

Your score will be calculated using [Kendall's tau](https://en.wikipedia.org/wiki/Kendall_rank_correlation_coefficient), a correlation measure for ordinal data.

*This function should return a Series of length 500 and dtype int.*

In [11]:
import pandas as pd

doc = []
with open('assets/dates.txt') as file:
    for line in file:
        doc.append(line)

df = pd.Series(doc)
df.head(10)

0         03/25/93 Total time of visit (in minutes):\n
1                       6/18/85 Primary Care Doctor:\n
2    sshe plans to move as of 7/8/71 In-Home Servic...
3                7 on 9/27/75 Audit C Score Current:\n
4    2/6/96 sleep studyPain Treatment Pain Level (N...
5                    .Per 7/06/79 Movement D/O note:\n
6    4, 5/18/78 Patient's thoughts about current su...
7    10/24/89 CPT Code: 90801 - Psychiatric Diagnos...
8                         3/7/86 SOS-10 Total Score:\n
9             (4/10/71)Score-1Audit C Score Current:\n
dtype: object

In [15]:
def date_sorter():
    import regex as re
    order = None
# YOUR CODE HERE

    pattern1 = r'(?P<text>(?P<month>\d?\d)[/|-](?P<day>\d?\d)[/|-](?P<year>\d{4}))'
    pattern2 = r'(?P<text>(?P<month>\d?\d)[/|-](?P<day>[0-2]?([0-9]|[3][01]))[/|-](?P<year>\d{2}))'
    pattern3 = r'(?P<text>(?P<day>\d?\d) ?(?P<month>[a-zA-Z]{3,})\.?,? (?P<year>\d{4}))'
    pattern4 = r'(?P<text>(?P<month>[a-zA-Z]{3,})\.?-? ?(?P<day>\d\d?)([th|nd|st])?,?-? ?(?P<year>\d{4}))' # 3+ digit alphabetical month + single or 2 digit month +optional st, nd, st
    pattern5 = '(?P<text>(?P<month>[A-Z][a-z]{2,}),?\.? (?P<year>\d{4}))' # 2+ digit alphabetical month  + 4 digit year
    pattern6=r'(?P<text>(?P<month>\d\d?)/(?P<year>\d{4}))'
    pattern7 = r'(?P<text>(?P<year>\d{4}))'

    list_of_patterns = [pattern1, pattern2, pattern3, pattern4, pattern5, pattern6, pattern7]
    df_working = pd.DataFrame()

    #print(len(level1_index_remaining))
    df_index_list = [ x for x in df.index.values]
    index_remaining =df.index.values

    for pattern in list_of_patterns:

        df_temp =df.loc[index_remaining].str.extractall(pat = pattern)
        df_working = pd.concat([df_working, df_temp], axis =0)

        index_found = [x[0] for x in df_working.index.values]

        index_remaining = [ x for x in df_index_list  if x not in index_found]

        print(len(df_working), 500- len(index_remaining))

    print(len(df_working))    
    
    convert_year = lambda  x : '19'+ str(x)  if len(x) ==2 else x
    df_working['year'] = df_working['year'].apply(convert_year)
    df_working['day'] = df_working['day'].fillna(value = 1)
    df_working['day'].value_counts()

    df_working['month'] = df_working['month'].fillna(value = 1)

    # normalized year
    df_working['year'] = df_working['year'].apply(lambda x: '19' + x if len(x) == 2 else x)
    df_working['year'] = df_working['year'].apply(lambda x: str(x))

    # normalized month
    df_working['month'] = df_working['month'].apply(lambda x: x[1:] if type(x) is str and x.startswith('0') else x)
    
    print(df_working[df_working['month']=='Age']['day'])
    
    df_working.loc[df_working['month']=='Age', 'day'] = '1'
    print(df_working[df_working['month']=='Age']['day'])
    
    month_dict = dict({'September': 9, 'Mar': 3, 'November': 11, 'Jul': 7, 'January': 1, 'December': 12,
                       'Feb': 2, 'May': 5, 'Aug': 8, 'Jun': 6, 'Sep': 9, 'Oct': 10, 'June': 6, 'March': 3,
                       'February': 2, 'Dec': 12, 'Apr': 4, 'Jan': 1, 'Janaury': 1,'August': 8, 'October': 10,
                       'July': 7, 'Since': 1, 'Nov': 11, 'April': 4, 'Decemeber': 12, 'Age': 1})
    df_working.replace({"month": month_dict}, inplace=True)
    df_working['month'] = df_working['month'].apply(lambda x: str(x))

    # Day
    df_working['day'] = df_working['day'].apply(lambda x: str(x))
        # Cleaned date
    df_working['date'] = df_working['month'] + '/' + df_working['day'] + '/' + df_working['year']
    df_working['date'] = pd.to_datetime(df_working['date'])
    
    idx = df_working.index
    #print(type(idx))   
    #print(isinstance(idx, pd.MultiIndex))
    if  isinstance(idx, pd.MultiIndex):
        idx.rename(names = ['orig_index', 'match'], inplace = True)
        df_working.reset_index(level = 0, inplace = True)
    
    #display(df_working.head(5))
    df_working.sort_values(by=['date','orig_index'], ascending = [True, True,], inplace=True)
    
    #df_working
    #df_working.rename(index={0:'Date'}, inplace = True)
    if df_working.index.name == 'match':
        df_working.reset_index(inplace = True)
    
    #display(df_working)     
    
    from IPython.display import display, HTML
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
        display(HTML(df_working.to_html()))
    
#     with pd.ExcelWriter(path = 'solution.xls') as writer:
#         df_working.to_excel(writer, sheet_name = 'df_working')
#         df.to_excel(writer, sheet_name = 'df')
        
    #df1 = pd.Series([x[0] for x  in df_working.index.values])
    df1 = df_working['orig_index']
    return df1


In [16]:
a=date_sorter()

25 25
125 125
195 195
230 230
345 345
457 457
500 500
500
     match
461  0        16
Name: day, dtype: object
     match
461  0        1
Name: day, dtype: object


,match,orig_index,text,month,day,year,3,date
0,0,9,4/10/71,4,10,1971,0,1971-04-10
1,0,84,5/18/71,5,18,1971,8,1971-05-18
2,0,2,7/8/71,7,8,1971,8,1971-07-08
3,0,53,7/11/71,7,11,1971,1,1971-07-11
4,0,28,9/12/71,9,12,1971,2,1971-09-12
5,0,474,1972,1,1,1972,NaN,1972-01-01
6,0,153,13 Jan 1972,1,13,1972,NaN,1972-01-13
7,0,13,1/26/72,1,26,1972,6,1972-01-26
8,0,129,06 May 1972,5,06,1972,NaN,1972-05-06
9,0,98,5/13/72,5,13,1972,3,1972-05-13


In [17]:
# produce some test checksums
# if you get for example a False entry in the agree column for
# index value 20 that would mean you have at least one incorrectly
# parsed date in the **output** index range 20,21,...,29
import numpy as np
s = date_sorter()
print(pd.DataFrame({'correct':
[6695, 14428, 16742, 9275, 12290, 14654, 9421, 10185, 11464, 16491,
 11797, 14036, 15459, 9412, 13069, 10400, 10498, 14322, 13274, 11001,
 11383, 11910, 10977, 9692, 10199, 10187, 15456, 13491, 9186, 13646,
 11142, 13724, 10994, 12905, 15968, 16648, 13966, 14607, 16932, 14622,
 17942, 18220, 17818, 18305, 19633, 12522, 13978, 18445, 20156, 14797],
'learner':[
(s.iloc[10*i:(i+1)*10].values * np.array(range(1,11))).sum() for i in range(50)]},
index=range(0,500,10)).assign(agree=lambda x:x['correct']==x['learner']))

25 25
125 125
195 195
230 230
345 345
457 457
500 500
500
     match
461  0        16
Name: day, dtype: object
     match
461  0        1
Name: day, dtype: object


,match,orig_index,text,month,day,year,3,date
0,0,9,4/10/71,4,10,1971,0,1971-04-10
1,0,84,5/18/71,5,18,1971,8,1971-05-18
2,0,2,7/8/71,7,8,1971,8,1971-07-08
3,0,53,7/11/71,7,11,1971,1,1971-07-11
4,0,28,9/12/71,9,12,1971,2,1971-09-12
5,0,474,1972,1,1,1972,NaN,1972-01-01
6,0,153,13 Jan 1972,1,13,1972,NaN,1972-01-13
7,0,13,1/26/72,1,26,1972,6,1972-01-26
8,0,129,06 May 1972,5,06,1972,NaN,1972-05-06
9,0,98,5/13/72,5,13,1972,3,1972-05-13


     correct  learner  agree
0       6695     6695   True
10     14428    14428   True
20     16742    16742   True
30      9275     9275   True
40     12290    12290   True
50     14654    14654   True
60      9421     9421   True
70     10185    10185   True
80     11464    11464   True
90     16491    16491   True
100    11797    11797   True
110    14036    14036   True
120    15459    15459   True
130     9412     9412   True
140    13069    13069   True
150    10400    10400   True
160    10498    10498   True
170    14322    14322   True
180    13274    13274   True
190    11001    11001   True
200    11383    11383   True
210    11910    11910   True
220    10977    10977   True
230     9692     9692   True
240    10199    10199   True
250    10187    10187   True
260    15456    15456   True
270    13491    13491   True
280     9186     9186   True
290    13646    13646   True
300    11142    11142   True
310    13724    13724   True
320    10994    10994   True
330    12905  